In [1]:
import sqlite3
import os

def rename_column(db_path, old_col, new_col):
    # 检查文件是否存在
    if not os.path.exists(db_path):
        print(f"❌ 错误: 找不到文件 '{db_path}'")
        return

    try:
        # 连接数据库
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        print(f"🔌 已连接到数据库: {db_path}")

        # 获取数据库中所有的表名
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()

        renamed_count = 0

        # 遍历所有表
        for table_name in tables:
            table = table_name[0]
            
            # 获取当前表的所有列信息
            cursor.execute(f"PRAGMA table_info({table})")
            columns_info = cursor.fetchall()
            
            # 提取列名列表
            columns = [col[1] for col in columns_info]

            # 检查 'industry' 是否在这个表中
            if old_col in columns:
                print(f"🔍 在表 '{table}' 中发现列 '{old_col}'...")
                
                # 执行重命名操作 (适用于 SQLite 3.25.0+)
                try:
                    query = f"ALTER TABLE {table} RENAME COLUMN {old_col} TO {new_col}"
                    cursor.execute(query)
                    print(f"✅ 成功: 表 '{table}' 的列已从 '{old_col}' 重命名为 '{new_col}'")
                    renamed_count += 1
                except sqlite3.OperationalError as e:
                    print(f"⚠️ 无法重命名表 '{table}' 中的列。原因: {e}")
            else:
                # 如果这个表里没有 target column，这就跳过
                pass

        # 提交更改并关闭连接
        if renamed_count > 0:
            conn.commit()
            print(f"💾 所有更改已保存。共修改了 {renamed_count} 个表。")
        else:
            print(f"ℹ️ 未找到名为 '{old_col}' 的列，没有进行任何修改。")

        conn.close()

    except Exception as e:
        print(f"❌ 发生意外错误: {e}")

if __name__ == "__main__":
    # 配置部分
    DB_FILE = 'testdb_cryptonews.db'
    OLD_COLUMN = 'industry'
    NEW_COLUMN = 'currency'

    # 执行函数
    rename_column(DB_FILE, OLD_COLUMN, NEW_COLUMN)

🔌 已连接到数据库: testdb_cryptonews.db
🔍 在表 'messages' 中发现列 'industry'...
✅ 成功: 表 'messages' 的列已从 'industry' 重命名为 'currency'
💾 所有更改已保存。共修改了 1 个表。
